# CAPSTONE PROJECT
## Irregular Migration Across the Western Balkan Route: The Case of Montenegro (2016–2026)

### Notebook 01 — Data Ingestion, Cleaning and Exploratory Data Analysis (EDA)

**Purpose:** prepare and explore the five selected sheets from the IOM DTM Europe Q1 2026 workbook.

**Important interpretation:** `Arrivals` represents recorded arrivals/events as defined by the source; it should not automatically be interpreted as a count of unique individuals. `Individuals present` is a different measure and must not be added to `Arrivals`.

The workbook contains nine sheets. This notebook loads the five selected analytical sheets only.


## 1. Import libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid')

## 2. Load the five selected sheets

In [ ]:
file_path = '/content/DTM Europe_Q1_2026_Dataset.xlsx'

selected_sheets = [
    'First_Arrival_Countries_Monthly',
    'Other_Countries_Areas_Monthly',
    'First_Arrival_Countries_CoO',
    'Others_Countries_CoO',
    'All_Migrant_Presence'
]

datasets = {
    sheet: pd.read_excel(file_path, sheet_name=sheet)
    for sheet in selected_sheets
}

# Short names used throughout the notebook
im1_1 = datasets['First_Arrival_Countries_Monthly']
im1_2 = datasets['Other_Countries_Areas_Monthly']
im1_3 = datasets['First_Arrival_Countries_CoO']
im1_4 = datasets['Others_Countries_CoO']
im1_5 = datasets['All_Migrant_Presence']

print('Loaded sheets:', list(datasets.keys()))

## 3. Inspect dataset structure

In [ ]:
for name, data in datasets.items():
    print(f'\\n{"=" * 60}')
    print(f'DATASET: {name}')
    print('Shape:', data.shape)
    print('Columns:', data.columns.tolist())
    print('Data types:')
    print(data.dtypes)
    print('First rows:')
    display(data.head(3))

## 4. Check missing values and duplicate rows

In [ ]:
quality_report = []

for name, data in datasets.items():
    quality_report.append({
        'Dataset': name,
        'Rows': len(data),
        'Columns': data.shape[1],
        'Duplicate rows': int(data.duplicated().sum()),
        'Missing cells': int(data.isna().sum().sum())
    })

display(pd.DataFrame(quality_report))

for name, data in datasets.items():
    missing = pd.DataFrame({
        'Missing values': data.isna().sum(),
        'Missing percentage': (data.isna().mean() * 100).round(2)
    })
    print(f'\\n{name}')
    display(missing)

## 5. Prepare date and numeric columns

Convert date columns to datetime and `Arrivals` to numeric. Invalid values are converted to missing values and reported; they are not silently replaced with zero.

In [ ]:
for name, data in datasets.items():
    if 'Date' in data.columns:
        data['Date'] = pd.to_datetime(data['Date'], errors='coerce')

    if 'Arrivals' in data.columns:
        data['Arrivals'] = pd.to_numeric(data['Arrivals'], errors='coerce')

    datasets[name] = data

# Refresh the short references after preparation
im1_1 = datasets['First_Arrival_Countries_Monthly']
im1_2 = datasets['Other_Countries_Areas_Monthly']
im1_3 = datasets['First_Arrival_Countries_CoO']
im1_4 = datasets['Others_Countries_CoO']
im1_5 = datasets['All_Migrant_Presence']

for name, data in datasets.items():
    if 'Date' in data.columns:
        print(name, '| invalid/missing dates:', data['Date'].isna().sum())
    if 'Arrivals' in data.columns:
        print(name, '| missing Arrivals:', data['Arrivals'].isna().sum())

## 6. Check country codes and route labels

In [ ]:
for name, data in datasets.items():
    print(f'\\n{name}')
    for column in data.columns:
        if column.startswith('ISO3'):
            codes = sorted(data[column].dropna().astype(str).unique())
            print(column, '| unique codes:', len(codes), '| Montenegro (MNE):', 'MNE' in codes)

print('\\nRoutes in First_Arrival_Countries_Monthly:')
print(im1_1['Route'].value_counts(dropna=False))

print('\\nRoutes in Other_Countries_Areas_Monthly:')
print(im1_2['Route'].value_counts(dropna=False))

## 7. Explore first-arrival routes

This sheet contains route categories such as EMR, WMR, CMR and WAAR in the notebook's inspected output. It did **not** contain Montenegro in `ISO3 - Country of arrival`, so it is not the source for the Montenegro Western Balkan Route target.

In [ ]:
route_first_arrival = (
    im1_1.groupby(['Year', 'Route'], as_index=False)['Arrivals']
    .sum()
    .sort_values('Year')
)

plt.figure(figsize=(12, 6))
sns.lineplot(
    data=route_first_arrival,
    x='Year',
    y='Arrivals',
    hue='Route',
    marker='o'
)
plt.title('Recorded Arrivals by First-Arrival Route')
plt.xlabel('Year')
plt.ylabel('Recorded arrivals')
plt.tight_layout()
plt.show()

print('Arrival countries in im1_1:')
print(sorted(im1_1['ISO3 - Country of arrival'].dropna().unique()))

## 8. Explore Other Countries/Areas and prepare the main route series

The inspected notebook output shows `WBR` and `EU Other` as route labels in this sheet. The following creates a monthly total for WBR across the countries represented in this sheet.

In [ ]:
western_balkan_route_name = 'WBR'

df_wb = im1_2.copy()
df_wb = df_wb.dropna(subset=['Date', 'Arrivals'])
df_wb = df_wb[df_wb['Route'] == western_balkan_route_name].copy()

df_wb['Month_Start'] = df_wb['Date'].dt.to_period('M').dt.to_timestamp()

wb_monthly = (
    df_wb.groupby('Month_Start')['Arrivals']
    .sum()
    .sort_index()
)

display(wb_monthly.head())
display(wb_monthly.tail())
print('WBR monthly observations:', len(wb_monthly))
print('Period:', wb_monthly.index.min(), 'to', wb_monthly.index.max())

In [ ]:
plt.figure(figsize=(14, 6))
plt.plot(wb_monthly.index, wb_monthly.values, label='Western Balkan Route')
plt.title('Monthly Recorded Arrivals — Western Balkan Route')
plt.xlabel('Month')
plt.ylabel('Recorded arrivals')
plt.legend()
plt.tight_layout()
plt.show()

## 9. Prepare the Montenegro Western Balkan Route series

**Important correction:** the earlier exploratory code filtered only `MNE`, so it could include records from more than one route. For a Montenegro-specific Western Balkan Route series, filter by **both** `Route == 'WBR'` and `ISO3 - Country of arrival == 'MNE'`.

In [ ]:
df_mne_wbr = im1_2.copy()
df_mne_wbr = df_mne_wbr.dropna(subset=['Date', 'Arrivals'])

df_mne_wbr = df_mne_wbr[
    (df_mne_wbr['Route'] == western_balkan_route_name) &
    (df_mne_wbr['ISO3 - Country of arrival'] == 'MNE')
].copy()

df_mne_wbr['Month_Start'] = (
    df_mne_wbr['Date'].dt.to_period('M').dt.to_timestamp()
)

mne_wbr_monthly = (
    df_mne_wbr.groupby('Month_Start')['Arrivals']
    .sum()
    .sort_index()
)

display(mne_wbr_monthly.head())
display(mne_wbr_monthly.tail())
print('Montenegro WBR monthly observations:', len(mne_wbr_monthly))
print('Period:', mne_wbr_monthly.index.min(), 'to', mne_wbr_monthly.index.max())
print('Total recorded arrivals in this filtered series:', mne_wbr_monthly.sum())

In [ ]:
plt.figure(figsize=(14, 6))
plt.plot(
    mne_wbr_monthly.index,
    mne_wbr_monthly.values,
    marker='o',
    label='Montenegro — WBR'
)
plt.title('Monthly Recorded Arrivals — Montenegro, Western Balkan Route')
plt.xlabel('Month')
plt.ylabel('Recorded arrivals')
plt.legend()
plt.tight_layout()
plt.show()

## 10. Explore countries of origin

The two CoO sheets contain origin and arrival country fields. They can support a separate descriptive analysis of origin-country patterns. Before using them for a forecast, verify their date coverage, definitions and overlap with the monthly sheets. Do not add their `Arrivals` values to the monthly sheet totals.

In [ ]:
print('Others_Countries_CoO date range:')
print(im1_4['Date'].min(), 'to', im1_4['Date'].max())

print('Routes in Others_Countries_CoO:')
print(im1_4['Route'].value_counts(dropna=False))

mne_origin = im1_4[
    (im1_4['Route'] == western_balkan_route_name) &
    (im1_4['ISO3 - Country of arrival'] == 'MNE')
].copy()

origin_summary = (
    mne_origin.groupby('ISO3 - Country of origin', as_index=False)['Arrivals']
    .sum()
    .sort_values('Arrivals', ascending=False)
)

display(origin_summary.head(15))

## 11. Explore migrant presence separately

`All_Migrant_Presence` uses `Individuals present`, not `Arrivals`. The inspected notebook output covers weekly observations in 2026. Keep this as a separate descriptive measure; do not combine it with monthly arrival counts.

In [ ]:
mne_presence = im1_5[
    im1_5['ISO3 - Host country'] == 'MNE'
].copy()

display(mne_presence.sort_values('Date'))
print('Presence date range:', mne_presence['Date'].min(), 'to', mne_presence['Date'].max())

## 12. EDA handover — what is ready for Modeling?

The primary forecasting inputs prepared in this notebook are:

- `wb_monthly`: monthly recorded arrivals for the WBR route across the countries represented in `Other_Countries_Areas_Monthly`.
- `mne_wbr_monthly`: monthly recorded arrivals for Montenegro (`MNE`) on the WBR route only.

The source workbook inspected in this notebook is the **Q1 2026** release, and the displayed WBR series ends in March 2026. Therefore, any forecast from this workbook begins after the last observed month, subject to the actual data coverage.

Before modeling, check that every month is represented and determine whether any missing month means zero recorded arrivals or unavailable data. Do not automatically fill missing months with zero.

**Limitations:** recorded arrivals are not necessarily unique people; the presence measure is different; historical patterns cannot account for sudden policy, conflict, border or other external changes unless those variables are included.